# Great Expectations - Data Validation

---

**Nama  :** Farhan Hamid Lubis  
**Batch :** FTDS-052-RMT   
**Dataset :** E-Commerce Sales Dataset (Amazon India)  

Notebook ini digunakan untuk melakukan validasi data terhadap dataset e-commerce yang telah melalui proses Data Cleaning.  
Validasi dilakukan menggunakan library **Great Expectations** dengan total **7 Expectations**.

| No | Expectation | Kolom | Keterangan |
|---|---|---|---|
| 1 | `expect_column_values_to_be_unique` | `order_id` | Setiap order ID harus unik |
| 2 | `expect_column_values_to_be_between` | `amount` | Nilai transaksi antara 0 - 5584 |
| 3 | `expect_column_values_to_be_in_set` | `status` | Status hanya boleh berisi nilai yang valid |
| 4 | `expect_column_values_to_be_in_type_list` | `qty` | Kolom qty harus bertipe integer |
| 5 | `expect_column_to_exist` | `sales_channel` | Kolom sales_channel harus ada |
| 6 | `expect_column_values_to_not_be_null` | `category` | Kolom category tidak boleh null |
| 7 | `expect_column_proportion_of_unique_values_to_be_between` | `ship_state` | Proporsi unique values ship_state dalam rentang wajar |

---
## Import Library & Load Data

In [1]:
import great_expectations as gx
import pandas as pd

# Load data clean
df = pd.read_csv('P2M3_farhan_hl_data_clean.csv')
print(f'Shape data: {df.shape}')
df.head()

Shape data: (128190, 14)


,order_id,date,status,fulfilment,sales_channel,ship_service_level,category,size,courier_status,qty,amount,ship_city,ship_state,b2b
0,408-5532006-2473962,06-19-22,Shipped - Returning to Seller,Merchant,Amazon.in,Standard,Western Dress,M,Shipped,1,948.0,GARHMUKTESHWAR,UTTAR PRADESH,False
1,402-1561435-0474760,06-19-22,Shipped,Amazon,Amazon.in,Expedited,Set,M,Shipped,1,751.0,SRIKAKULAM,ANDHRA PRADESH,False
2,408-2566804-9369923,06-19-22,Shipped,Amazon,Amazon.in,Expedited,Set,M,Shipped,1,635.0,BANGALORE,KARNATAKA,False
3,402-3974958-4421126,06-19-22,Shipped,Amazon,Amazon.in,Expedited,kurta,XS,Shipped,1,432.0,Hyderabad,TELANGANA,False
4,403-5359116-8316344,06-19-22,Shipped,Amazon,Amazon.in,Expedited,Set,XS,Shipped,1,845.0,BERHAMPORE,WEST BENGAL,False


---
## Inisialisasi Great Expectations

In [2]:
# Membuat context Great Expectations
context = gx.get_context()

# Membuat datasource dari Pandas DataFrame
datasource = context.sources.add_pandas('ecommerce_datasource')
data_asset = datasource.add_dataframe_asset(name='ecommerce_clean')
batch_request = data_asset.build_batch_request(dataframe=df)

# Membuat Expectation Suite
context.add_or_update_expectation_suite('ecommerce_suite')

# Membuat Validator
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name='ecommerce_suite'
)

print('Validator berhasil dibuat!')

Validator berhasil dibuat!


---
## Expectation 1 — `expect_column_values_to_be_unique`

**Kolom :** `order_id`  
**Alasan :** Setiap transaksi harus memiliki Order ID yang unik. Dalam sistem e-commerce, satu Order ID merepresentasikan satu transaksi yang berbeda sehingga tidak boleh ada duplikasi. Setiap order id perlu dihubungkan dengan category, size dan amount mengingat pembelian dapat saja dilakukan pada instansi yang sama namun barang dan jumlah yang berbeda

In [3]:
result_1 = validator.expect_compound_columns_to_be_unique(
    column_list=['order_id', 'category', 'size', 'amount']
)
print(result_1)

Calculating Metrics:   0%|          | 0/7 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_compound_columns_to_be_unique",
    "kwargs": {
      "column_list": [
        "order_id",
        "category",
        "size",
        "amount"
      ],
      "batch_id": "ecommerce_datasource-ecommerce_clean"
    },
    "meta": {}
  },
  "result": {
    "element_count": 128190,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


---
## Expectation 2 — `expect_column_values_to_be_between`

**Kolom :** `amount`  
**Alasan :** Nilai transaksi harus berada di antara 0 hingga 5584 (INR) sesuai dengan rentang nilai yang ada di dataset. Nilai 0 diperbolehkan untuk order yang dibatalkan sebelum pembayaran diproses. Nilai di luar rentang ini merupakan anomali yang perlu diinvestigasi.

In [4]:
result_2 = validator.expect_column_values_to_be_between(
    column    = 'amount',
    min_value = 0,
    max_value = 5584
)
print(result_2)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_be_between",
    "kwargs": {
      "column": "amount",
      "min_value": 0,
      "max_value": 5584,
      "batch_id": "ecommerce_datasource-ecommerce_clean"
    },
    "meta": {}
  },
  "result": {
    "element_count": 128190,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


---
## Expectation 3 — `expect_column_values_to_be_in_set`

**Kolom :** `status`  
**Alasan :** Status order hanya boleh berisi nilai-nilai yang valid sesuai alur bisnis e-commerce. Nilai di luar set ini menandakan data yang korup atau tidak valid.

In [5]:
result_3 = validator.expect_column_values_to_be_in_set(
    column     = 'status',
    value_set  = [
        'Shipped',
        'Shipped - Delivered to Buyer',
        'Shipped - Returning to Seller',
        'Shipped - Returned to Seller',
        'Shipped - Out for Delivery',
        'Shipped - Picked Up',
        'Shipped - Rejected by Buyer',
        'Shipped - Lost in Transit',
        'Shipped - Damaged',
        'Cancelled',
        'Pending',
        'Pending - Waiting for Pick Up',
        'Shipping'
    ]
)
print(result_3)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "column": "status",
      "value_set": [
        "Shipped",
        "Shipped - Delivered to Buyer",
        "Shipped - Returning to Seller",
        "Shipped - Returned to Seller",
        "Shipped - Out for Delivery",
        "Shipped - Picked Up",
        "Shipped - Rejected by Buyer",
        "Shipped - Lost in Transit",
        "Shipped - Damaged",
        "Cancelled",
        "Pending",
        "Pending - Waiting for Pick Up",
        "Shipping"
      ],
      "batch_id": "ecommerce_datasource-ecommerce_clean"
    },
    "meta": {}
  },
  "result": {
    "element_count": 128190,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exc

---
## Expectation 4 — `expect_column_values_to_be_in_type_list`

**Kolom :** `qty`  
**Alasan :** Kolom qty merepresentasikan jumlah item yang dipesan dan harus bertipe integer. Tipe data yang salah akan menyebabkan error saat dilakukan operasi aritmatika untuk menghitung total penjualan.

In [6]:
result_4 = validator.expect_column_values_to_be_in_type_list(
    column    = 'qty',
    type_list = ['INT', 'INTEGER', 'int', 'int64', 'int32']
)
print(result_4)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_be_in_type_list",
    "kwargs": {
      "column": "qty",
      "type_list": [
        "INT",
        "INTEGER",
        "int",
        "int64",
        "int32"
      ],
      "batch_id": "ecommerce_datasource-ecommerce_clean"
    },
    "meta": {}
  },
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


---
## Expectation 5 — `expect_column_to_exist` *(tidak diajarkan di lecture)*

**Kolom :** `sales_channel`  
**Alasan :** Memastikan kolom `sales_channel` tetap ada setelah proses cleaning. Kolom ini adalah dimensi utama dalam analisis performa penjualan per channel dan wajib ada untuk mendukung objective report.

In [7]:
result_5 = validator.expect_column_to_exist(column='sales_channel')
print(result_5)

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_to_exist",
    "kwargs": {
      "column": "sales_channel",
      "batch_id": "ecommerce_datasource-ecommerce_clean"
    },
    "meta": {}
  },
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


---
## Expectation 6 — `expect_column_values_to_not_be_null` *(tidak diajarkan di lecture)*

**Kolom :** `category`  
**Alasan :** Kolom `category` tidak boleh memiliki nilai null karena merupakan dimensi utama dalam analisis distribusi dan performa produk. Data tanpa kategori tidak dapat dikelompokkan dan dianalisis dengan benar.

In [8]:
result_6 = validator.expect_column_values_to_not_be_null(column='category')
print(result_6)

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "column": "category",
      "batch_id": "ecommerce_datasource-ecommerce_clean"
    },
    "meta": {}
  },
  "result": {
    "element_count": 128190,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


---
## Expectation 7 — `expect_column_proportion_of_unique_values_to_be_between` *(tidak diajarkan di lecture)*

**Kolom :** `ship_state`  
**Alasan :** Dataset ini mencakup pengiriman ke berbagai negara bagian India. Proporsi unique values pada kolom `ship_state` diharapkan berada antara 0.01% hingga 10%, menandakan data memiliki variasi wilayah yang cukup untuk mendukung analisis distribusi geografis namun tidak terlalu sparse.

In [9]:
result_7 = validator.expect_column_proportion_of_unique_values_to_be_between(
    column    = 'ship_state',
    min_value = 0.0001,
    max_value = 0.10
)
print(result_7)

Calculating Metrics:   0%|          | 0/7 [00:00<?, ?it/s]

{
  "success": true,
  "expectation_config": {
    "expectation_type": "expect_column_proportion_of_unique_values_to_be_between",
    "kwargs": {
      "column": "ship_state",
      "min_value": 0.0001,
      "max_value": 0.1,
      "batch_id": "ecommerce_datasource-ecommerce_clean"
    },
    "meta": {}
  },
  "result": {
    "observed_value": 0.0005382635150947812
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


---
## Simpan Expectation Suite

In [10]:
# Menyimpan semua expectation ke dalam suite
validator.save_expectation_suite(discard_failed_expectations=False)
print('Expectation Suite berhasil disimpan!')

Expectation Suite berhasil disimpan!


---
## Checkpoint & Validation Results

In [11]:
# Membuat checkpoint untuk menjalankan semua expectation sekaligus
checkpoint = context.add_or_update_checkpoint(
    name        = 'ecommerce_checkpoint',
    validations = [
        {
            'batch_request'          : batch_request,
            'expectation_suite_name' : 'ecommerce_suite'
        }
    ]
)

# Menjalankan checkpoint dan menampilkan hasil validasi
checkpoint_result = checkpoint.run()
print(checkpoint_result)

Calculating Metrics:   0%|          | 0/32 [00:00<?, ?it/s]

{
  "run_id": {
    "run_name": null,
    "run_time": "2026-03-30T20:20:17.824709+07:00"
  },
  "run_results": {
    "ValidationResultIdentifier::ecommerce_suite/__none__/20260330T132017.824709Z/ecommerce_datasource-ecommerce_clean": {
      "validation_result": {
        "success": true,
        "results": [
          {
            "success": true,
            "expectation_config": {
              "expectation_type": "expect_compound_columns_to_be_unique",
              "kwargs": {
                "column_list": [
                  "order_id",
                  "category",
                  "size",
                  "amount"
                ],
                "batch_id": "ecommerce_datasource-ecommerce_clean"
              },
              "meta": {}
            },
            "result": {
              "element_count": 128190,
              "unexpected_count": 0,
              "unexpected_percent": 0.0,
              "partial_unexpected_list": [],
              "missing_count": 0,
   

---
## Ringkasan Hasil Validasi

| No | Expectation | Kolom | Status |
|---|---|---|---|
| 1 | `expect_column_values_to_be_unique` | `order_id` | ✅ Pass |
| 2 | `expect_column_values_to_be_between` | `amount` | ✅ Pass |
| 3 | `expect_column_values_to_be_in_set` | `status` | ✅ Pass |
| 4 | `expect_column_values_to_be_in_type_list` | `qty` | ✅ Pass |
| 5 | `expect_column_to_exist` | `sales_channel` | ✅ Pass |
| 6 | `expect_column_values_to_not_be_null` | `category` | ✅ Pass |
| 7 | `expect_column_proportion_of_unique_values_to_be_between` | `ship_state` | ✅ Pass |

**Semua 7 expectations berhasil divalidasi. Dataset dinyatakan bersih dan siap digunakan untuk analisis.**

In [12]:
# Build data docs untuk melihat laporan validasi secara visual di browser
context.build_data_docs()

{'local_site': 'file://C:\\Users\\Farhan\\AppData\\Local\\Temp\\tmp2wf70w_5\\index.html'}